In [ ]:
%load_ext autoreload
%autoreload 2

from arch.univariate.base import ARCHModelResult


In [ ]:
from mock_alpha import activate_mock
activate_mock()


In [ ]:
# Import your libraries here
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from arch import arch_model
from config import settings
from data import SQLRepository
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import requests
import plotly.express as px


In [ ]:
ticker = "MTNOY"
output_size = "full"
data_type = "json"

url = (
    "https://learn-api.wqu.edu/1/data-services/alpha-vantage/query?"
    "function=TIME_SERIES_DAILY&"
    f"symbol={ticker}&"
    f"outputsize={output_size}&"
    f"datatype={data_type}&"
    f"apikey=572f37a53a7b1ffc1333ce0711c18542886ee7a2c24c697978d22dd8e5fb1c7027c9abd72d7eb3656bb9a43e6fed88e719352d1281797210639c3fe3c7481a592ada63af1a3544c96ec85a32408258cf2ec42da685163ba1714b0ffa16b08c2e6bfb4e82f73130250f8aa24632b0f8ad1a38767310ce60ca735b7a17ec841e"
)

print("url type:", type(url))
url


In [ ]:
response = requests.get(url=url)

print("response type:", type(response))


In [ ]:
response_code = response.status_code

print("code type:", type(response_code))
response_code


In [ ]:
response_data = response.json()
stock_data = response_data["Time Series (Daily)"]

df_mtnoy = pd.DataFrame.from_dict(
    stock_data,
    orient="index",
    dtype=float
)

df_mtnoy.index = pd.to_datetime(df_mtnoy.index)
df_mtnoy.index.name = "date"

df_mtnoy.columns = [c.split(". ")[1] for c in df_mtnoy.columns]

print("df_mtnoy type:", type(df_mtnoy))
df_mtnoy.head()


In [ ]:
connection = sqlite3.connect(database=settings.db_name,check_same_thread=False)
connection


In [ ]:
# Insert `MTNOY` data into database
from data import SQLRepository

repo = SQLRepository(connection=connection)

response = repo.insert_table(
    table_name=ticker,
    records=df_mtnoy,
    if_exists="replace"
)


In [ ]:
sql = "SELECT * FROM 'MTNOY'"

df_mtnoy_read = pd.read_sql(
    sql=sql,
    con=connection,
    parse_dates=["date"],
    index_col="date"
)

print("df_mtnoy_read type:", type(df_mtnoy_read))
print("df_mtnoy_read shape:", df_mtnoy_read.shape)

df_mtnoy_read.head()


In [ ]:
def wrangle_data(ticker, n_observations):
    df = repo.read_table(
        table_name=ticker,
        limit=n_observations + 1
    )

    df.sort_index(ascending=True, inplace=True)

    df["return"] = df["close"].pct_change() * 100

    return df["return"].dropna()


y_mtnoy = wrangle_data(
    ticker="MTNOY",
    n_observations=2500
)

y_mtnoy.head()


In [ ]:
mtnoy_daily_volatility = y_mtnoy.std()

print("mtnoy_daily_volatility type:", type(mtnoy_daily_volatility))
print("MTN Daily Volatility:", mtnoy_daily_volatility)


In [ ]:
mtnoy_annual_volatility = mtnoy_daily_volatility * np.sqrt(252)

print("mtnoy_annual_volatility type:", type(mtnoy_annual_volatility))
print("MTN Annual Volatility:", mtnoy_annual_volatility)


In [ ]:
# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Plot `y_mtnoy` on `ax`
y_mtnoy.plot(ax=ax, label="daily return")

# Add axis labels
plt.xlabel("Date")
plt.ylabel("Returns")

# Add title
plt.title("Time Series of MTNOY Returns")


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Create ACF of squared returns
plot_acf(y_mtnoy**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient")

# Add title
plt.title("ACF of MTNOY Squared Returns")


In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Create PACF of squared returns
plot_pacf(y_mtnoy**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient")

# Add title
plt.title("PACF of MTNOY Squared Returns")


In [ ]:
cutoff_test = int(len(y_mtnoy) * 0.8)

y_mtnoy_train = y_mtnoy.iloc[:cutoff_test]

print("y_mtnoy_train type:", type(y_mtnoy_train))
print("y_mtnoy_train shape:", y_mtnoy_train.shape)

y_mtnoy_train.head()


In [ ]:
# Build and train model
model = arch_model(
    y_mtnoy_train,
    p=1,
    q=1,
    rescale=False
).fit(disp=0)

print("model type:", type(model))

# Show model summary
print(model.summary())


In [ ]:
# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Plot standardized residuals
model.std_resid.plot(ax=ax, label="Standardized Residuals")

# Add axis labels and title
plt.xlabel("Date")
plt.ylabel("Value")
plt.title("MTNOY GARCH Model Standardized Residuals")


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Create ACF of squared, standardized residuals
plot_acf(model.std_resid**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient")

# Add title
plt.title("ACF of MTNOY GARCH Model Standardized Residuals")


In [ ]:
# Import `build_model` function
from main import build_model

# Build model using new `MTNOY` data
model = build_model(ticker="MTNOY", use_new_data=True)

# Wrangle `MTNOY` returns
model.wrangle_data(n_observations=2500)

# Fit GARCH(1,1) model to data
model.fit(p=1, q=1)

# Does model have AIC and BIC attributes?
assert hasattr(model, "aic")
assert hasattr(model, "bic")


In [ ]:
# Import `FitIn` class and `fit_model` function
from main import FitIn, fit_model

# Instantiate `FitIn` object
request = FitIn(ticker="MTNOY", use_new_data=False, n_observations=2500, p=1, q=1)

# Build model and fit to data, following parameters in `request`
fit_out = fit_model(request=request)

# Inspect `fit_out`
fit_out


In [ ]:
# URL of `/fit` path
url = "http://localhost:8008/fit"

# Data to send to path
json = {
    "ticker": "MTNOY",
    "use_new_data": False,
    "n_observations": 2500,
    "p": 1,
    "q": 1
}

# Response of post request
response = requests.post(url=url, json=json)

print("response type:", type(response))
print("response status code:", response.status_code)


In [ ]:
# URL of `/predict` path
url = "http://localhost:8008/predict"

# Data to send to path
json = {
    "ticker": "MTNOY",
    "n_days": 5
}

# Response of post request
response = requests.post(url=url, json=json)

print("response type:", type(response))
print("response status code:", response.status_code)
